# Canonical Model Template

This notebook is a readable scaffold for building a canonical `myflopy` model from GeoPackage inputs. It intentionally leaves project-specific inputs as placeholders so the model can be shaped from a real workflow instead of from synthetic assumptions.

The important API pattern is:

1. Define paths and model timing.
2. Build or load the grid, surfaces, and `idomain`.
3. Create one `ModelContext`.
4. Build ordinary boundary packages from GeoPackages.
5. Build advanced packages with configured builders and zero-argument `.build()` calls.
6. Assemble `ModelSpec` and `SimulationSpec`.
7. Write, run, and review outputs.


## Expected Input Layers

Replace these with your actual GeoPackage paths and field names. The field names shown here are recommended placeholders, not requirements.

- `chd.gpkg`: `name`, `layer`, `head_0`, `head_1`, ...
- `ghb.gpkg`: `name`, `layer`, `head`, `conductance`, or relative fields such as `height_above_bottom`, `min_elev`
- `drn.gpkg`: `name`, `layer`, `elevation`, `conductance`, or relative fields such as `drain_offset`, `min_elev`
- `wel.gpkg`: `name`, `layer`, `rate_0`, `rate_1`, ...
- `rch.gpkg`: `name`, `layer`, `recharge_0`, `recharge_1`, ...
- `lakes.gpkg`: `lake_id`, `starting_stage`, `lake_bottom`, `bed_leakance`
- `streams.gpkg`: `stream_id`, `width`, `reach_top`, `gradient`, `roughness`, `streambed_k`, `streambed_thickness`
- optional `k_zones.gpkg`: `name`, `layer`, `k`


In [1]:
from pathlib import Path

import flopy
import geopandas as gpd
import numpy as np
import pandas as pd

import myflopy as mf


In [2]:
# Project and run configuration. Replace these paths first.
project_root = Path("../artifacts/canonical_template")
run_workspace = project_root / "baseline"
input_root = project_root / "inputs"

model_name = "canonical_gwf"
simulation_name = "canonical_simulation"
nlay = 3
nper = 13

perioddata = [(30.0, 10, 1.1)] * nper

inputs = {
    "chd": input_root / "chd.gpkg",
    "ghb": input_root / "ghb.gpkg",
    "drn": Path(r"C:\Users\lukem\mf6\Cumberland general\Boundaries\bak\drn_bak.gpkg"),
    "wel": input_root / "wel.gpkg",
    "rch": input_root / "rch.gpkg",
    "lakes": input_root / "lakes.gpkg",
    "streams": input_root / "streams.gpkg",
    "k_zones": input_root / "k_zones.gpkg",
}


## Grid, Surfaces, And Domain

This is the one section that should match your normal grid workflow. The rest of the notebook assumes these objects exist:

- `vor`: a `VoronoiGridPlus` or compatible grid helper
- `top`: one value per cell
- `botm`: one list/array per layer, shape `(nlay, ncpl)`
- `idomain`: active-domain array, shape `(nlay, ncpl)`
- `starting_heads`: initial heads, shape `(nlay, ncpl)` or FloPy-compatible equivalent


In [3]:

deep_lake_path = Path(r"C:\Users\lukem\mf6\Cumberland general\Boundaries\deep lake.gpkg")
hyde_lake_path = Path(r"C:\Users\lukem\mf6\Cumberland general\Boundaries\hyde lake.gpkg")
deep_creek = Path(r"C:\Users\lukem\mf6\Cumberland general\Boundaries\deep creek.gpkg")
hyde_creek = Path(r"C:\Users\lukem\mf6\Cumberland general\Boundaries\hyde creek.gpkg")
top_r = Path(r"C:\Users\lukem\mf6\Cumberland general\Surfaces\lidar_top_of_model_existing_v5.tif")
botm_r = Path(r"C:\Users\lukem\mf6\Cumberland general\Surfaces\cumb_aq_btm_v14c.tif")
domain_v3 = Path(r"C:\Users\lukem\mf6\Cumberland general\domain_v3.gpkg")

tri = mf.TriangleGrid(model_ws=str(run_workspace / 'triangle'), angle=30)
tri.set_domain_file(shp_gpkg=Path(r"C:\Users\lukem\mf6\Cumberland general\domain_v3.gpkg"), max_area=50_000, simplify_tolerance=100, buffer=-100)
tri.add_line_buffer(deep_creek, buffer=20, simplify_tolerance=10, densify_dist=200, max_area=300, domain=domain_v3)
tri.add_line_buffer(hyde_creek, buffer=20, simplify_tolerance=10, densify_dist=200, max_area=300, domain=domain_v3)
tri.add_polygon(deep_lake_path, simplify_tolerance=20, max_area=5000)
tri.add_polygon(hyde_lake_path, simplify_tolerance=20, max_area=5000)
tri.build()

vor = mf.VoronoiGridPlus(tri=tri, crs='EPSG:2926', rasters=[top_r, botm_r, ])

top = ...
botm = ...
idomain = ...
starting_heads = np.load(r"C:\Users\lukem\Python\Projects\simple_modflow\examples\mf6\artifacts\canonical_template\starting_heads.npy")

# Optional but useful: keep surfaces on the grid for builders that can use them.
# vor.gdf_topbtm = gpd.GeoDataFrame(..., geometry=vor.gdf_vorPolys.geometry, crs=vor.crs)


Imported 1 features from C:\Users\lukem\mf6\Cumberland general\domain_v3.gpkg
Imported 1 features from C:\Users\lukem\mf6\Cumberland general\Boundaries\deep creek.gpkg
Imported 1 features from C:\Users\lukem\mf6\Cumberland general\Boundaries\hyde creek.gpkg
Imported 1 features from C:\Users\lukem\mf6\Cumberland general\Boundaries\deep lake.gpkg
Imported 1 features from C:\Users\lukem\mf6\Cumberland general\Boundaries\hyde lake.gpkg
VoronoiGrid initializing.
Voronoi grid initialized.


In [4]:
vor.gdf_topbtm[2] = vor.gdf_topbtm[1] - 20.0
vor.gdf_topbtm[3] = vor.gdf_topbtm[1] - 40.0
vor.gdf_topbtm.loc[:, 0:3] = vor.reconcile_surfaces()

# FloPy DISV wants plain array-like values, not pandas Series/DataFrames.
# Column 0 is model top; columns 1..nlay are layer bottoms.
surface_columns = list(range(1, nlay + 1))
top = vor.gdf_topbtm[0].to_numpy(dtype=float).tolist()
botm = [vor.gdf_topbtm[column].to_numpy(dtype=float).tolist() for column in surface_columns]

ncpl = len(top)
idomain = np.ones((nlay, ncpl), dtype=int)


reading raster file C:\Users\lukem\mf6\Cumberland general\Surfaces\lidar_top_of_model_existing_v5.tif
reading raster file C:\Users\lukem\mf6\Cumberland general\Surfaces\cumb_aq_btm_v14c.tif


In [5]:
context = mf.ModelContext(
    grid=vor,
    surfaces=getattr(vor, "gdf_topbtm", None),
    domain=idomain,
    metadata={
        "model_name": model_name,
        "template": "canonical_model_template",
    },
)

gridprops = vor.get_gridprops_vertexgrid()
ncpl = int(vor.ncpl)


## Core MODFLOW Packages

These are ordinary `PackageSpec` objects around FloPy 3.10 constructors. Keep these simple and explicit; the goal is to make the model build readable rather than clever.


In [6]:
disv = mf.disv(
    nlay=nlay,
    ncpl=ncpl,
    nvert=len(gridprops["vertices"]),
    vertices=gridprops["vertices"],
    cell2d=gridprops["cell2d"],
    top=top,
    botm=botm,
)

ic = mf.ic(strt=starting_heads)

# Replace these hydraulic-property placeholders with arrays or scalar values.
k = 500
k33 = 100
icelltype = [1] * nlay

npf = mf.npf(
    k=k,
    k33=k33,
    icelltype=icelltype,
    save_flows=True,
    save_specific_discharge=True,
)

sto = mf.sto(
    ss=1.0e-5,
    sy=0.15,
    iconvert=icelltype,
    steady_state={0: True},
    transient={period: True for period in range(1, nper)},
    save_flows=True,
)

oc = mf.oc(
    head_filerecord=f"{model_name}.hds",
    budget_filerecord=f"{model_name}.cbc",
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
)


## GeoPackage Boundary Packages

Package-first helpers map GIS features to grid cells using the shared `ModelContext`. Use separate paths or GeoPackage layers when packages live in different files or layers.


In [7]:
# Uncomment packages as the corresponding input files and fields are ready.
# chd = mf.chd.gpkg(
#     inputs["chd"],
#     context=context,
#     nper=nper,
#     head=[f"head_{period}" for period in range(nper)],
#     edges_only=True,
# )

# ghb = mf.ghb.gpkg(
#     inputs["ghb"],
#     context=context,
#     nper=nper,
#     head=mf.CellSurfaceOffset(
#         reference="cell_bottom",
#         offset="height_above_bottom",
#         minimum="min_elev",
#     ),
#     conductance="conductance",
# )

drn = mf.drn.gpkg(
    inputs["drn"],
    context=context,
    nper=nper,
    elevation=mf.CellSurfaceOffset(reference="cell_top", offset=0),
    conductance="cond",
)

# wel = mf.wel.gpkg(
#     inputs["wel"],
#     context=context,
#     nper=nper,
#     rate=[f"rate_{period}" for period in range(nper)],
# )

# Option 1: recharge zones from GIS attributes.
# rch = mf.rch.gpkg(
#     inputs["rch"],
#     context=context,
#     nper=nper,
#     recharge=[f"recharge_{period}" for period in range(nper)],
# )

# Option 2: recharge defined directly from the model domain.
rch = mf.rch(
    context=context,
    nper=nper,
    recharge=0.00001,
    cells="top_active",
)

# Option 3: direct FloPy-style RCH stress-period data.
# rch = mf.rch.flopy(stress_period_data={0: [[(0, 10), 1.0e-4]]})

# Cell-specific recharge can use (layer, cell) keys.
# cell_specific_rch = mf.rch(
#     context=context,
#     nper=nper,
#     recharge={
#         (0, 10): 1.0e-4,
#         (0, 11): 1.2e-4,
#     },
# )


## Advanced Packages

These builders keep all configuration on the builder object. Their `.build()` methods take no configuration. Use `with_updates(...)` to create scenario variants.


In [ ]:
# UZF: package-first for ordinary use, builder object when you want to inspect/edit before building.
finf = {period: 0.006 for period in range(nper)}
pet = {period: 0.001 for period in range(nper)}

# uzf = mf.uzf(
#     context=context,
#     nper=nper,
#     cells="all_active",
#     vks=...,
#     thtr=0.08,
#     thts=0.35,
#     thti=0.18,
#     eps=4.0,
#     finf=finf,
#     pet=pet,
#     extdp=None,
#     simulate_et=False,
# )

# LAK/SFR often stay as builders until MVR is declared, because their semantic
# connection helpers can be used to build mover endpoints.
lak_builder = mf.LAKBuilder(
    context=context,
    nper=nper,
    lakes=inputs["lakes"],
    lake_id_field="lake_id",
    starting_stage="starting_stage",
    lake_bottom="lake_bottom",
    bed_leakance="bed_leakance",
    connection_modes={
        "north_lake": "automatic",
        "south_lake": "automatic",
        # "infiltration_trench": "rectangular",
    },
    tables={
        # "north_lake": mf.LakeTableBuilder(dem=Path("..."), footprint=Path("..."), stages=np.arange(...)),
    },
    outlets=(
        # mf.LakeOutlet(source="north_lake", receiver="south_lake", rate={0: 0.0}),
    ),
    status="ACTIVE",
    mover=True,
    length_conversion=3.28081,
    time_conversion=86_400.0,
)

sfr_builder = mf.SFRBuilder(
    context=context,
    nper=nper,
    streams=inputs["streams"],
    stream_id="stream_id",
    connection_mode="automatic",
    connection_tolerance=...,
    connections=(
        # mf.StreamConnection(source="tributary_a", receiver="main_stem", source_location="downstream", receiver_location="nearest"),
    ),
    diversions=(
        # mf.StreamDiversion(source="main_stem", receiver="canal", amount={0: 0.25}, priority="FRACTION"),
    ),
    width="width",
    gradient="gradient",
    reach_top="reach_top",
    roughness="roughness",
    streambed_k="streambed_k",
    streambed_thickness="streambed_thickness",
    inflow={0: [(0, ...)]},
    mover=True,
    length_conversion=3.28081,
    time_conversion=86_400.0,
)


## MVR Coupling

Build MVR after creating LAK and SFR builders, because mover endpoints can be requested from those builders by stable feature ID. If you already have direct MVR period data, use `mf.mvr.flopy(...)` instead.


In [ ]:
mvr_builder = mf.MVRBuilder(
    nper=nper,
    moves={
        period: [
            mf.Move(
                source=sfr_builder.connection("main_stem", location="downstream"),
                receiver=lak_builder.connection("north_lake"),
                method="FACTOR",
                value=0.25,
            ),
            # mf.Move(source=lak_builder.connection("south_lake"), receiver=sfr_builder.connection("main_stem"), value=0.05),
        ]
        for period in range(nper)
    },
)

mvr_builder.perioddata


## Optional Observation Hook

Keep observation wiring separate from package construction. A post-build hook receives the completed FloPy model, the built package dictionary, and the `ModelContext`.


In [ ]:
def attach_observations(model, packages, context):
    """Attach observations after all packages exist."""

    # TODO: add HeadTargets, LakeStageTargets, SfrStageTargets,
    # SfrFlowTargets, and DrnFlowTargets for the canonical model.
    return {"attached": False, "todo": "define canonical target registry"}


observation_hook = mf.PostBuildHook("observations", attach_observations)


## Assemble The Model And Simulation

Package order matters when one package depends on another. `MVRBuilder.build()` declares dependencies on the packages referenced by its moves, so put it after `lak_builder.build()` and `sfr_builder.build()`.


In [8]:
"""flow = mf.ModelSpec(
    model_name,
    "gwf",
    context=context,
    packages=(
        disv,
        ic,
        npf,
        sto,
        # chd,
        # ghb,
        drn,
        # wel,
        rch,
        # lak_builder.build(),
        # sfr_builder.build(),
        # uzf,
        # mvr_builder.build(),
        oc,
    ),
    # hooks=(observation_hook,),
)"""

flow = mf.gwf(
    model_name,
    context=context,
    save_flows=True,
    packages=(disv, ic, npf, sto, drn, rch, oc),
    newtonoptions='newton',
)

simulation = mf.SimulationSpec(
    simulation_name,
    models=(flow,),
    packages=(
        mf.tdis(nper=nper, perioddata=perioddata),
        mf.ims(
            models=(model_name,),
            print_option="SUMMARY",
            complexity="COMPLEX",
            outer_maximum=1200,
            inner_maximum=300,
            outer_dvclose=1e-3,
            inner_dvclose=1e-4,
            rcloserecord=[1e-3, "strict"],
            linear_acceleration="BICGSTAB",
            under_relaxation="DBD",
            under_relaxation_momentum=0.001,
            under_relaxation_theta=0.7,
            under_relaxation_kappa=0.1,
            under_relaxation_gamma=0.2,
            backtracking_number=50,
            backtracking_tolerance=1.2,
            backtracking_reduction_factor=0.2,
            backtracking_residual_limit=10.0,
        )
    ),
)




## Build, Write, Run

Keep build/write/run separate while developing. It makes it easier to inspect generated package files before spending time on a model run.


In [9]:
run = simulation.build(run_workspace)

model = run.model()

In [10]:
run.write()
success, report = run.execute(silent=False)

FloPy is using the following executable to run the model: ..\..\..\..\..\..\..\mf-env\.venv\Scripts\mf6.exe
                               MODFLOW 6 EXTENDED
                U.S. GEOLOGICAL SURVEY MODULAR HYDROLOGIC MODEL
                        VERSION 6.8.0.dev0+cb8a12e.dirty
                               ***DEVELOP MODE***

   MODFLOW 6 compiled Jun 07 2026 15:32:35 with Intel(R) Fortran Intel(R) 64
  Compiler Classic for applications running on Intel(R) 64, Version 2021.12.0
                             Build 20240222_000000

This software is preliminary or provisional and is subject to 
revision. It is being provided to meet the need for timely best 
science. The software has not received final approval by the U.S. 
Geological Survey (USGS). No warranty, expressed or implied, is made 
by the USGS or the U.S. Government as to the functionality of the 
software and related material nor shall the fact of release 
constitute any such warranty. The software is provided on the 
conditi

In [ ]:
start_hds: np.ndarray = model.hds.get_alldata()[9]

[## Scenario Variants

Because builders are configured values, variants can be made by changing a builder and rebuilding its package spec. This keeps scenario changes local and visible.


In [ ]:
np.save(file=r"C:\Users\lukem\Python\Projects\simple_modflow\examples\mf6\artifacts\canonical_template\starting_heads", arr=start_hds)

In [ ]:
len(model.hds.get_alldata()[9])

In [ ]:
run.simulation
model.summary()

In [ ]:
# If you need scenario variants for UZF, keep the explicit builder around:
# uzf_builder = mf.UZFBuilder(...)
# higher_recharge_uzf = uzf_builder.with_updates(
#     finf={period: ... for period in range(nper)},
# )
# variant_flow = flow.with_package(higher_recharge_uzf.build())
# variant_simulation = simulation.with_model(variant_flow)

# variant_run = variant_simulation.build(project_root / "higher_recharge")
# variant_run.write()
# variant_model = variant_run.model()
